In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!pip install -q huggingface_hub datasets

In [ ]:
import gc, json, time, threading, subprocess, psutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from huggingface_hub import hf_hub_download
from datasets import load_dataset
from tqdm.auto import tqdm


try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
    print("[INFO] Google Drive mounted.")
except ImportError:
    IN_COLAB = False
    print("[WARN] Not in Colab.")

_DRIVE_ROOT = "/content/drive/MyDrive/tarecmind"

VARIANT_NAME = "lightgcn_baseline"

CFG = {
    "REPO_ID":        "chuongdo1104/amazon-2023-gold",
    "SILVER_REPO_ID": "chuongdo1104/amazon-2023-silver",
    "DRIVE_ROOT":     _DRIVE_ROOT,
    "DATA_DIR":       f"{_DRIVE_ROOT}/data_{VARIANT_NAME}",
    "SAVE_DIR":       f"{_DRIVE_ROOT}/weights_{VARIANT_NAME}",
    "CACHE_DIR":      "/content/recsys_cache",
    "EMBED_DIM":      128,
    "GCN_LAYERS":     2,
    "EPOCHS":         50,
    "LR_JOINT":       1e-3,
    "L2_REG":        1e-4,
    "WEIGHT_DECAY":   0.0,   
    "GRAD_CLIP":      1.0,
    "PATIENCE":       5,
    "EVAL_EVERY":     2,
    "KEEPALIVE_MINS": 25,
}

for d in [CFG["DATA_DIR"], CFG["SAVE_DIR"], CFG["CACHE_DIR"]]:
    os.makedirs(d, exist_ok=True)

print(f"[INFO] Variant: {VARIANT_NAME}")

Mounted at /content/drive
[INFO] Google Drive mounted.
[INFO] Variant: lightgcn_baseline


In [3]:
def detect_and_adjust_gpu():
    sys_ram_gb = psutil.virtual_memory().total / 1e9
    print(f"[INFO] System RAM: {sys_ram_gb:.1f} GB")
    if not torch.cuda.is_available():
        print("[WARN] No GPU.")
        return "cpu"
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[INFO] GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")
    if vram_gb > 70:
        CFG["BATCH_SIZE"]  = 32768
        CFG["ACCUM_STEPS"] = 1
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
    elif vram_gb > 35:
        CFG["BATCH_SIZE"]  = 16384
        CFG["ACCUM_STEPS"] = 2
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
    elif vram_gb > 16:
        CFG["BATCH_SIZE"]  = 4096
        CFG["ACCUM_STEPS"] = 4
    else:
        CFG["BATCH_SIZE"]  = 2048
        CFG["ACCUM_STEPS"] = 4
    print(f"   -> batch={CFG['BATCH_SIZE']}, accum={CFG['ACCUM_STEPS']}")
    return "cuda"

device = detect_and_adjust_gpu()

_keepalive_stop = threading.Event()
def _keepalive_worker():
    interval = CFG["KEEPALIVE_MINS"] * 60
    ka_file  = os.path.join(CFG["CACHE_DIR"], "keepalive.txt")
    while not _keepalive_stop.is_set():
        time.sleep(interval)
        if not _keepalive_stop.is_set():
            with open(ka_file, "w") as f:
                f.write(f"alive {time.strftime('%H:%M:%S')}\n")
threading.Thread(target=_keepalive_worker, daemon=True).start()
print(f"[INFO] Keep-alive started. Device: {device}")

[INFO] System RAM: 89.6 GB
[INFO] GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4 GB
   -> batch=16384, accum=2
[INFO] Keep-alive started. Device: cuda


In [4]:
POP_GROUP  = {0: "HEAD", 1: "MID", 2: "TAIL", 3: "COLD_START"}
def download_hf(filename, local_dir=None):
    target     = local_dir or CFG["CACHE_DIR"]
    local_path = os.path.join(target, os.path.basename(filename))
    if os.path.exists(local_path):
        return local_path
    print(f"[INFO] Downloading {filename}...")
    return hf_hub_download(
        repo_id=CFG["REPO_ID"], filename=filename,
        repo_type="dataset", local_dir=target,
    )

print("\n--- LOADING GOLD METADATA ---")
with open(download_hf("gold/gold_dataset_stats.json"), "r") as f:
    dataset_stats = json.load(f)

num_users   = dataset_stats["n_users"]
num_items   = dataset_stats["n_items"]
total_nodes = num_users + num_items
print(f"[INFO] {num_users:,} users | {num_items:,} items | {total_nodes:,} total nodes")
print(f"[INFO] Sparsity: {dataset_stats.get('sparsity_pct','N/A')}")

edge_index_raw   = np.load(download_hf("gold/gold_edge_index.npy"))
train_edge_index = torch.from_numpy(edge_index_raw).long()
n_train_edges    = train_edge_index.shape[1]
del edge_index_raw

item_train_freq_np = np.load(download_hf("gold/gold_item_train_freq.npy"))
item_pop_group_np  = np.load(download_hf("gold/gold_item_popularity_group.npy"))
item_train_freq_t  = torch.from_numpy(item_train_freq_np).long()
item_pop_group_t   = torch.from_numpy(item_pop_group_np.astype("int32")).long()

assert n_train_edges > 0,              "Edge list empty!"
assert len(item_train_freq_np) == num_items

# ============================================================
# ALL-ITEM WITH COLD SETUP:
# Warm item = item có ít nhất 1 tương tác trong train.
# Cold item = item có train_freq == 0.
# Evaluation giữ cả warm và cold trong validation/test positives
# và trong candidate ranking.
# Negative sampling khi train vẫn chỉ lấy warm item để không coi
# cold item là negative training signal.
# Tail/Cold được báo cáo riêng để phân tích long-tail và cold-start.
# ============================================================
item_train_freq_cpu = item_train_freq_t.detach().cpu()
warm_item_mask_cpu  = item_train_freq_cpu > 0
cold_item_mask_cpu  = ~warm_item_mask_cpu
warm_item_ids_cpu   = torch.where(warm_item_mask_cpu)[0].long()
num_warm_items      = int(warm_item_mask_cpu.sum().item())
num_cold_items      = int(cold_item_mask_cpu.sum().item())

assert num_warm_items > 0, "No warm items found!"
assert num_warm_items + num_cold_items == num_items

print(f"\n[INFO] {n_train_edges:,} train edges")
for code, name in POP_GROUP.items():
    cnt = int((item_pop_group_t == code).sum())
    warm_cnt = int(((item_pop_group_t == code).cpu() & warm_item_mask_cpu).sum())
    print(f"  {name:10s}: {cnt:,} total | {warm_cnt:,} warm")
cnt_cold = int((item_pop_group_t == 3).sum())
print(f"  {'COLD_START':10s}: {cnt_cold:,} total | 0 warm (freq=0 by definition)")
print(f"[INFO] All-item catalog: {num_items:,} items | warm={num_warm_items:,} | cold kept={num_cold_items:,}")

del item_train_freq_np, item_pop_group_np
gc.collect()
print("[OK] Gold data loaded. Warm/cold masks prepared.")


--- LOADING GOLD METADATA ---
[INFO] Downloading gold/gold_dataset_stats.json...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


gold_dataset_stats.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

[INFO] 1,847,662 users | 1,610,012 items | 3,457,674 total nodes
[INFO] Sparsity: 99.9995%
[INFO] Downloading gold/gold_edge_index.npy...


gold/gold_edge_index.npy:   0%|          | 0.00/223M [00:00<?, ?B/s]

[INFO] Downloading gold/gold_item_train_freq.npy...


gold/gold_item_train_freq.npy:   0%|          | 0.00/12.9M [00:00<?, ?B/s]

[INFO] Downloading gold/gold_item_popularity_group.npy...


gold/gold_item_popularity_group.npy:   0%|          | 0.00/1.61M [00:00<?, ?B/s]


[INFO] 13,964,281 train edges
  HEAD      : 210,925 total | 210,925 warm
  MID       : 116,283 total | 116,283 warm
  TAIL      : 714,913 total | 714,913 warm
  COLD_START: 567,891 total | 0 warm
  COLD_START: 567,891 total | 0 warm (freq=0 by definition)
[INFO] All-item catalog: 1,610,012 items | warm=1,042,121 | cold kept=567,891
[OK] Gold data loaded. Warm/cold masks prepared.


In [5]:
path_train_edges = os.path.join(CFG["DATA_DIR"], "train_edges.pt")
path_val_edges   = os.path.join(CFG["DATA_DIR"], "val_edges.pt")
path_sparse_adj  = os.path.join(CFG["DATA_DIR"], "sparse_adj.pt")

# -- Val edges --
if os.path.exists(path_val_edges):
    print("[INFO] Loading val_edges from Drive cache...")
    val_edges_t = torch.load(path_val_edges, map_location="cpu", weights_only=True)
    VAL_SIZE    = val_edges_t.shape[1]
else:
    print("[INFO] Building val_edges from silver_val_ground_truth.parquet...")
    df_val_gt = load_dataset(
        CFG["SILVER_REPO_ID"],
        data_dir="silver/silver_val_ground_truth.parquet",
        split="train",
    ).to_pandas()
    path_item_map = download_hf("gold/gold_item_id_map.parquet")
    path_user_map = download_hf("gold/gold_user_id_map.parquet")
    df_item_map = pd.read_parquet(path_item_map, columns=["parent_asin", "item_idx"])
    df_user_map = pd.read_parquet(path_user_map, columns=["reviewer_id",  "user_idx"])
    df_val_gt = (
        df_val_gt
        .merge(df_item_map, on="parent_asin", how="inner")
        .merge(df_user_map, on="reviewer_id",  how="inner")
    )
    VAL_SIZE    = len(df_val_gt)
    val_edges_t = torch.tensor(
        np.stack([df_val_gt["user_idx"].values.astype(np.int64),
                  df_val_gt["item_idx"].values.astype(np.int64)], axis=0),
        dtype=torch.long,
    )
    torch.save(val_edges_t, path_val_edges)
    del df_val_gt, df_item_map, df_user_map; gc.collect()
print(f"  val_edges raw: {VAL_SIZE:,}")

# Val positives: giữ nguyên tất cả, kể cả cold items.
# Cold items sẽ xuất hiện trong COLD group khi evaluate.
VAL_SIZE = int(val_edges_t.shape[1])
val_cold_cnt = int((item_train_freq_cpu[val_edges_t[1].cpu()] == 0).sum())
print(f"[INFO] Val: {VAL_SIZE:,} total | cold positives: {val_cold_cnt:,}")

# -- Train edges --
if os.path.exists(path_train_edges):
    train_edge_index_dev = torch.load(path_train_edges, map_location=device, weights_only=True)
else:
    train_edge_index_dev = train_edge_index.to(device)
    torch.save(train_edge_index_dev.cpu(), path_train_edges)
train_edge_index = train_edge_index_dev
n_train_edges    = train_edge_index.shape[1]

# -- Sparse adj (D^-0.5 A D^-0.5) --
if os.path.exists(path_sparse_adj):
    print("[INFO] Loading sparse_adj from Drive cache...")
    sparse_adj = torch.load(path_sparse_adj, map_location=device, weights_only=True)
else:
    print("[INFO] Building sparse_adj...")
    row = torch.cat([train_edge_index[0], train_edge_index[1] + num_users])
    col = torch.cat([train_edge_index[1] + num_users, train_edge_index[0]])
    gei = torch.stack([row, col]).long()
    deg = torch.bincount(gei[0], minlength=total_nodes).float()
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
    ew  = (deg_inv_sqrt[gei[0]] * deg_inv_sqrt[gei[1]]).half()
    adj = torch.sparse_coo_tensor(gei, ew, (total_nodes, total_nodes))
    sparse_adj = adj.coalesce().to(device).to_sparse_csr()
    torch.save(sparse_adj.cpu(), path_sparse_adj)
    del adj, ew, deg, deg_inv_sqrt, gei, row, col; gc.collect()

# -- Move tensors and masks to device --
item_train_freq_t = item_train_freq_t.to(device)
item_pop_group_t  = item_pop_group_t.to(device)
warm_item_mask    = warm_item_mask_cpu.to(device)
cold_item_mask    = cold_item_mask_cpu.to(device)
warm_item_ids     = warm_item_ids_cpu.to(device)

print(f"[OK] Graph ready. Train: {n_train_edges:,} | Val: {VAL_SIZE:,} (gồm cả cold)")
print(f"[OK] Candidate set all-item with cold: {num_items:,} items | warm={num_warm_items:,} | cold kept={num_cold_items:,}")
gc.collect(); torch.cuda.empty_cache()

[INFO] Loading val_edges from Drive cache...
  val_edges raw: 1,847,662
[INFO] Val: 1,847,662 total | cold positives: 72,685
[INFO] Loading sparse_adj from Drive cache...
[OK] Graph ready. Train: 13,964,281 | Val: 1,847,662 (gồm cả cold)
[OK] Candidate set all-item with cold: 1,610,012 items | warm=1,042,121 | cold kept=567,891


/usr/local/lib/python3.12/dist-packages/torch/_utils.py:371: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  result = torch.sparse_compressed_tensor(
/usr/local/lib/python3.12/dist-packages/torch/_utils.py:371: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  result = torch.sparse_compressed_tensor(


In [6]:
class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=128, gcn_layers=2):
        super().__init__()
        self.num_users  = num_users
        self.num_items  = num_items
        self.embed_dim  = embed_dim
        self.gcn_layers = gcn_layers
        self.user_id_emb = nn.Embedding(num_users, embed_dim)
        self.item_id_emb = nn.Embedding(num_items, embed_dim)
        nn.init.normal_(self.user_id_emb.weight, std=0.01)
        nn.init.normal_(self.item_id_emb.weight, std=0.01)

    def forward_gcn(self, adj):
        x = torch.cat([self.user_id_emb.weight, self.item_id_emb.weight], dim=0)
        z_accum = x.clone()
        for _ in range(self.gcn_layers):
            x = torch.sparse.mm(adj, x.to(adj.dtype)).to(torch.float32)
            z_accum += x
        return z_accum / (self.gcn_layers + 1.0)

    def forward(self, adj, batch_users, batch_pos, batch_neg):
        z_G = self.forward_gcn(adj)
        user_G, item_G = z_G.split([self.num_users, self.num_items])
        return user_G[batch_users], item_G[batch_pos], item_G[batch_neg]

print("[OK] LightGCN model class defined.")

[OK] LightGCN model class defined.


In [7]:
def bpr_loss(user_emb, pos_emb, neg_emb):
    pos_scores = (user_emb * pos_emb).sum(dim=1)   # dot product thô
    neg_scores = (user_emb * neg_emb).sum(dim=1)
    return -F.logsigmoid(pos_scores - neg_scores).mean()


def l2_reg_loss(model, users, pos_items, neg_items):
    """
    Manual L2 regularization for LightGCN.
    Regularize only the ego/id embeddings that appear in the current BPR batch,
    following the common LightGCN implementation style.
    """
    user_ego = model.user_id_emb(users)
    pos_ego  = model.item_id_emb(pos_items)
    neg_ego  = model.item_id_emb(neg_items)

    reg = (
        user_ego.norm(2).pow(2)
        + pos_ego.norm(2).pow(2)
        + neg_ego.norm(2).pow(2)
    )
    return 0.5 * reg / users.size(0)


print("[OK] BPR loss + manual L2 regularization defined.")


[OK] BPR loss defined.


In [8]:
def sample_negatives_uniform(batch_size, warm_item_ids, device):
    """
    PureCF training negative sampling: sample only warm items.
    Chỉ lấy item âm trong warm catalog, không lấy cold item.
    """
    if not isinstance(warm_item_ids, torch.Tensor):
        warm_item_ids = torch.tensor(warm_item_ids, dtype=torch.long, device=device)
    warm_item_ids = warm_item_ids.to(device)
    rand_pos = torch.randint(0, warm_item_ids.numel(), (batch_size,), device=device)
    return warm_item_ids[rand_pos]

_neg = sample_negatives_uniform(256, warm_item_ids, device)
assert _neg.shape == (256,)
assert (_neg >= 0).all() and (_neg < num_items).all()
assert (item_train_freq_t[_neg] > 0).all(), "Negative sampler still returns cold items!"
del _neg
print("[OK] Training negative sampler ready: warm items only; cold items kept for evaluation.")

[OK] Training negative sampler ready: warm items only; cold items kept for evaluation.


In [9]:
def evaluate_lightgcn(model, adj, item_pop_group_t, eval_groups,
                       num_users, num_items, Ks=(20, 40), device="cuda",
                       user_chunk=64, train_mask_dict=None,
                       warm_item_mask=None):
    """
    Full-ranking INTERACTION-LEVEL validation.

    Protocol:
    - Mỗi (user, positive_item) là một mẫu đánh giá độc lập.
    - Recall@K  = 1 nếu positive item trong top-K, else 0. Trung bình qua interactions.
    - NDCG@K    = 1/log2(rank+1) nếu rank<=K, else 0.    Trung bình qua interactions.
    - TAIL/COLD : lọc interactions có positive item thuộc nhóm tương ứng.
    - Candidate : toàn bộ catalog (kể cả cold items).
    - Mask       : train items bị loại khỏi ranking.
    """
    model.eval()
    Ks     = sorted(list(Ks))
    maxK   = max(Ks)
    pop_g  = item_pop_group_t.to(device)
    POP_NAMES   = {2: "TAIL", 3: "COLD"}
    group_names = ["OVERALL", "TAIL", "COLD"]

    accum = {
        g: {K: {"hit": 0.0, "ndcg": 0.0, "n": 0} for K in Ks}
        for g in group_names
    }
    recommended_mask = {
        K: torch.zeros(num_items, dtype=torch.bool, device=device) for K in Ks
    }

    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            z_G_all = model.forward_gcn(adj)
        user_G_all, item_G_all = z_G_all.split([num_users, num_items])
        item_final = item_G_all

    mask_value = torch.finfo(torch.float32).min

    for gname, gdata in eval_groups.items():
        if gdata is None:
            continue
        g_u = gdata["u"].to(device)
        g_i = gdata["i"].to(device)
        g_n = len(g_u)

        with torch.no_grad():
            with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                for start in tqdm(range(0, g_n, user_chunk),
                                  desc=f"Eval {gname}", ncols=100, leave=False):
                    end = min(start + user_chunk, g_n)
                    bu  = g_u[start:end]
                    bi  = g_i[start:end]

                    scores = torch.mm(user_G_all[bu], item_final.t())

                    # Mask train items per user
                    if train_mask_dict is not None:
                        for j, uid in enumerate(bu.cpu().numpy()):
                            mi = train_mask_dict.get(int(uid))
                            if mi is not None:
                                scores[j, mi.to(device)] = mask_value

                    # Rank của positive item (1-indexed)
                    pos_sc = scores[torch.arange(len(bu), device=device), bi]
                    ranks  = (scores > pos_sc.unsqueeze(1)).sum(dim=1) + 1

                    # Coverage mask
                    _, topk_idx = torch.topk(scores, maxK, dim=1)
                    for K in Ks:
                        recommended_mask[K][topk_idx[:, :K].reshape(-1)] = True

                    # Tích lũy metrics
                    bi_g = pop_g[bi]
                    for K in Ks:
                        hm = (ranks <= K).float()
                        nm = torch.where(
                            ranks <= K,
                            1.0 / torch.log2(ranks.float() + 1.0),
                            torch.zeros_like(ranks.float())
                        )
                        accum["OVERALL"][K]["hit"]  += hm.sum().item()
                        accum["OVERALL"][K]["ndcg"] += nm.sum().item()
                        accum["OVERALL"][K]["n"]    += len(bu)

                        for code, gn in POP_NAMES.items():
                            m = (bi_g == code)
                            if m.any():
                                accum[gn][K]["hit"]  += hm[m].sum().item()
                                accum[gn][K]["ndcg"] += nm[m].sum().item()
                                accum[gn][K]["n"]    += m.sum().item()

    # Aggregate
    results = {}
    for grp in group_names:
        results[grp] = {}
        for K in Ks:
            n = accum[grp][K]["n"]
            results[grp][f"Recall@{K}"] = accum[grp][K]["hit"]  / max(n, 1)
            results[grp][f"NDCG@{K}"]   = accum[grp][K]["ndcg"] / max(n, 1)
            results[grp][f"n@{K}"]      = n
        k0 = Ks[0]
        results[grp]["Recall@K"] = results[grp][f"Recall@{k0}"]
        results[grp]["NDCG@K"]   = results[grp][f"NDCG@{k0}"]
        results[grp]["n"]        = results[grp][f"n@{k0}"]

    # Coverage (tail = group 2, trên toàn bộ catalog)
    tail_mask_dev = (pop_g == 2)
    n_tail = int(tail_mask_dev.sum().item())
    cov, tcov, apop = {}, {}, {}
    for K in Ks:
        rm  = recommended_mask[K]
        nr  = int(rm.sum().item())
        cov[K]  = nr / num_items if num_items > 0 else 0
        tcov[K] = (rm & tail_mask_dev).sum().item() / n_tail if n_tail > 0 else 0
        apop[K] = item_train_freq_t[rm].float().mean().item() if nr > 0 else 0

    results["Coverage@K"]       = cov[Ks[0]]
    results["TailCoverage@K"]   = tcov[Ks[0]]
    results["AvgPopularity@K"]  = apop[Ks[0]]
    results["CoverageByK"]      = cov
    results["TailCoverageByK"]  = tcov
    results["AvgPopularityByK"] = apop
    results["EvalLevel"]        = "interaction-level"
    return results


def print_eval_results(epoch, results, Ks=(20, 40)):
    print(f"\n[VAL INTERACTION-LEVEL] Epoch {epoch:02d}")
    for grp in ["COLD", "TAIL", "OVERALL"]:
        if grp not in results:
            continue
        v = results[grp]
        n = v.get(f"n@{Ks[0]}", v.get("n", 0))
        parts = [f"  {grp:<10} | n={n:>10,}"]
        for K in Ks:
            parts.append(f"R@{K}={v.get(f'Recall@{K}', 0):.4f} | N@{K}={v.get(f'NDCG@{K}', 0):.4f}")
        print(" | ".join(parts))
    cov  = results.get("CoverageByK",  {})
    tcov = results.get("TailCoverageByK", {})
    apop = results.get("AvgPopularityByK", {})
    for K in Ks:
        print(f"  Coverage@{K}={cov.get(K,0):.4f} | TailCov@{K}={tcov.get(K,0):.4f} | AvgPop@{K}={apop.get(K,0):.1f}")
    tail = results.get("TAIL", {})
    cold = results.get("COLD", {})
    print(f"  [TAIL] R@20={tail.get('Recall@20',0):.4f} N@20={tail.get('NDCG@20',0):.4f} TailCov@20={tcov.get(20,0):.4f}")
    print(f"  [COLD] R@20={cold.get('Recall@20',0):.4f} N@20={cold.get('NDCG@20',0):.4f}")

print("[OK] Interaction-level LightGCN evaluation ready.")

[OK] Interaction-level LightGCN evaluation ready.


In [10]:
class ColabCheckpointManager:
    def __init__(self, weight_dir, data_dir, K=20):
        self.weight_dir = weight_dir
        self.data_dir   = data_dir
        self.K          = K
        os.makedirs(weight_dir, exist_ok=True)
        os.makedirs(data_dir, exist_ok=True)
        self.best_path = os.path.join(weight_dir, "best.pth")
        self.last_path = os.path.join(weight_dir, "last.pth")
        self.hist_path = os.path.join(data_dir,   "training_history.json")

    def save(self, model, optimizer, epoch, metrics, history_list, cfg, is_best=False):
        def _safe(v):
            return v if isinstance(v, (int,float,str,bool,type(None))) else str(v)
        safe_cfg = {k: _safe(v) for k,v in cfg.items() if not k.startswith("_")}
        payload = {
            "epoch":                epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "metrics":              metrics,
            "config":               safe_cfg,
        }
        torch.save(payload, self.last_path)
        if is_best:
            torch.save(payload, self.best_path)
            print(f"[CKPT] NEW BEST saved (Score={metrics.get('Score',0):.4f})")

        current = []
        if os.path.exists(self.hist_path):
            try:
                with open(self.hist_path, "r") as f: current = json.load(f)
            except Exception: pass
        existing_epochs = {h["epoch"] for h in current}
        for entry in history_list:
            if entry.get("epoch") not in existing_epochs:
                current.append(entry)
        current.sort(key=lambda x: x.get("epoch", 0))
        with open(self.hist_path, "w") as f:
            json.dump(current, f, indent=2, ensure_ascii=False)

    def try_resume(self, model, optimizer, device):
        if not os.path.exists(self.last_path):
            print("[CKPT] No checkpoint found. Starting from epoch 1.")
            return 0, 0.0, []
        ckpt = torch.load(self.last_path, map_location=device, weights_only=True)
        model.load_state_dict(ckpt["model_state_dict"])
        try: optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        except Exception as e: print(f"[WARN] optimizer state: {e}")
        start_epoch = ckpt.get("epoch", 0)
        history = []
        if os.path.exists(self.hist_path):
            try:
                with open(self.hist_path, "r") as f: history = json.load(f)
            except Exception: pass
        scores = [h.get("Score", 0.4*h.get("Recall@K", 0))
                  for h in history if "Score" in h or "Recall@K" in h]
        best_score = max(scores) if scores else 0.0
        print(f"[CKPT] Resume from epoch {start_epoch + 1}, best_score={best_score:.4f}")
        return start_epoch, best_score, history

ckpt_mgr = ColabCheckpointManager(weight_dir=CFG["SAVE_DIR"], data_dir=CFG["DATA_DIR"])
print("[OK] CheckpointManager ready.")

[OK] CheckpointManager ready.


In [11]:
def build_full_eval_groups(val_edges_t):
    """
    Full validation set — keeps original distribution.
    Used for early stopping and best checkpoint selection.
    """
    val_u = val_edges_t[0].cpu()
    val_i = val_edges_t[1].cpu()
    print(f"  [FULL VAL] {len(val_u):,} samples")
    return {"FULL": {"u": val_u, "i": val_i}}

print("[OK] Eval group builder ready.")

[OK] Eval group builder ready.


In [ ]:
for var in ["model", "optimizer", "scaler"]:
    if var in globals(): del globals()[var]
gc.collect(); torch.cuda.empty_cache()

model = LightGCN(
    num_users=num_users, num_items=num_items,
    embed_dim=CFG["EMBED_DIM"], gcn_layers=CFG["GCN_LAYERS"],
).to(device)

scaler = torch.amp.GradScaler("cuda", enabled=(device=="cuda"))
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CFG["LR_JOINT"],
    weight_decay=0.0,  
)
resume_epoch, best_score, history = ckpt_mgr.try_resume(model, optimizer, device)



print(f"[OK] LightGCN params: {sum(p.numel() for p in model.parameters()):,}")
gc.collect(); torch.cuda.empty_cache()

[CKPT] Resume from epoch 51, best_score=0.0107
[OK] LightGCN params: 442,582,272


In [13]:
print("[INFO] Building full validation groups for checkpoint selection...")
val_full_groups = build_full_eval_groups(val_edges_t)

print("[INFO] Building train mask dict...")
df_edges = pd.DataFrame({
    "u": train_edge_index[0].cpu().numpy(),
    "i": train_edge_index[1].cpu().numpy()
})
train_mask_dict = df_edges.groupby("u")["i"].apply(
    lambda x: torch.tensor(x.values, dtype=torch.long, device="cpu")
).to_dict()
del df_edges
print(f"[OK] Train mask for {len(train_mask_dict):,} users.")
gc.collect(); torch.cuda.empty_cache()

[INFO] Building full validation groups for checkpoint selection...
  [FULL VAL] 1,847,662 samples
[INFO] Building train mask dict...
[OK] Train mask for 1,847,662 users.


In [14]:
CFG["PATIENCE"]      = 5

# Do NOT override resume_epoch here — it is set by ckpt_mgr.try_resume().
# Overriding would reset training to epoch 1 even when a valid checkpoint exists.
if resume_epoch == 0:
    best_score = 0.0
    history    = []
patience_cnt = 0

if resume_epoch > 0:
    for g in optimizer.param_groups:
        g.setdefault('initial_lr', g['lr'])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["EPOCHS"], eta_min=1e-5, last_epoch=resume_epoch-1
)

steps_per_epoch = (n_train_edges + CFG["BATCH_SIZE"] - 1) // CFG["BATCH_SIZE"]
print(f"\n[INIT] Training LightGCN (Epoch {resume_epoch+1}/{CFG['EPOCHS']})")
print("[INFO] Each epoch uses torch.randperm — every train edge visited once per epoch.")

for epoch in range(resume_epoch, CFG["EPOCHS"]):
    t0 = time.time()
    model.train()
    total_loss = 0.0
    n_processed = 0

    perm = torch.randperm(n_train_edges, device=device)

    with tqdm(total=n_train_edges,
              desc=f"Ep {epoch+1:02d} [LightGCN BPR]", ncols=100) as pbar:
        optimizer.zero_grad(set_to_none=True)
        pending_accum_steps = 0

        for step, start in enumerate(range(0, n_train_edges, CFG["BATCH_SIZE"])):
            end = min(start + CFG["BATCH_SIZE"], n_train_edges)
            idx = perm[start:end]

            u = train_edge_index[0, idx]
            p = train_edge_index[1, idx]
            n = sample_negatives_uniform(len(u), warm_item_ids, device)

            with torch.amp.autocast("cuda", enabled=(device=="cuda")):
                f_u, f_p, f_n = model(sparse_adj, u, p, n)

                bpr = bpr_loss(f_u, f_p, f_n)
                reg = l2_reg_loss(model, u, p, n)
                raw_loss = bpr + CFG["L2_REG"] * reg

                loss = raw_loss / CFG["ACCUM_STEPS"]

            scaler.scale(loss).backward()
            pending_accum_steps += 1

            is_accum_boundary = (pending_accum_steps >= CFG["ACCUM_STEPS"])
            is_last_step      = (step == steps_per_epoch - 1)

            if is_accum_boundary or is_last_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                pending_accum_steps = 0

            total_loss  += raw_loss.item() * len(u)
            n_processed += len(u)
            pbar.update(len(u))

    scheduler.step()
    avg_loss = total_loss / max(n_processed, 1)
    print(f"[TRAIN] Ep {epoch+1:02d} | Loss={avg_loss:.4f} | SeenEdges={n_processed:,}/{n_train_edges:,}")

    metrics  = {}
    is_best  = False

    if (epoch+1) % CFG["EVAL_EVERY"] == 0:
        print("\n[VAL-FULL] Interaction-level early stopping and best checkpoint selection.")
        res_full = evaluate_lightgcn(
            model, sparse_adj, item_pop_group_t, val_full_groups,
            num_users, num_items, Ks=(20, 40), device=device,
            train_mask_dict=train_mask_dict,
            warm_item_mask=None  # cold items giữ trong candidate set
        )
        print_eval_results(epoch+1, res_full, Ks=(20, 40))

        overall_ndcg20 = res_full["OVERALL"]["NDCG@20"]
        score          = overall_ndcg20

        metrics = {
            "EvalLevel":             "interaction-level",
            "Recall@20":             res_full["OVERALL"]["Recall@20"],
            "NDCG@20":               res_full["OVERALL"]["NDCG@20"],
            "Recall@40":             res_full["OVERALL"]["Recall@40"],
            "NDCG@40":               res_full["OVERALL"]["NDCG@40"],
            "TailRecall@20":         res_full.get("TAIL", {}).get("Recall@20", 0),
            "TailNDCG@20":           res_full.get("TAIL", {}).get("NDCG@20", 0),
            "TailRecall@40":         res_full.get("TAIL", {}).get("Recall@40", 0),
            "TailNDCG@40":           res_full.get("TAIL", {}).get("NDCG@40", 0),
            "Score":                 score,
            "Coverage@20":           res_full.get("CoverageByK", {}).get(20, 0),
            "TailCoverage@20":       res_full.get("TailCoverageByK", {}).get(20, 0),
            "AvgPopularity@20":      res_full.get("AvgPopularityByK", {}).get(20, 0),
            "Coverage@40":           res_full.get("CoverageByK", {}).get(40, 0),
            "TailCoverage@40":       res_full.get("TailCoverageByK", {}).get(40, 0),
            "AvgPopularity@40":      res_full.get("AvgPopularityByK", {}).get(40, 0),
            "Recall@K":              res_full["OVERALL"]["Recall@20"],
            "NDCG@K":                res_full["OVERALL"]["NDCG@20"],
        }

        if score > best_score:
            best_score   = score
            is_best      = True
            patience_cnt = 0
            print(f"  NEW BEST: NDCG@20={score:.4f}")
        else:
            patience_cnt += 1
            print(f"  Patience {patience_cnt}/{CFG['PATIENCE']}")

    history.append({"epoch": epoch+1, "loss": avg_loss, **metrics})
    ckpt_mgr.save(model, optimizer, epoch+1, metrics, history, CFG, is_best=is_best)

    if patience_cnt >= CFG["PATIENCE"]:
        print(f"\n[STOP] Early stopping at epoch {epoch+1}.")
        break

print(f"\nTRAINING COMPLETE! Best NDCG@20={best_score:.4f}")



[INIT] Training LightGCN (Epoch 51/50)
[INFO] Each epoch uses torch.randperm — every train edge visited once per epoch.

TRAINING COMPLETE! Best NDCG@20=0.0107


In [15]:
gc.collect(); torch.cuda.empty_cache()

In [16]:
print("\n--- LOAD BEST MODEL ---")
if os.path.exists(ckpt_mgr.best_path):
    ckpt = torch.load(ckpt_mgr.best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"[OK] Loaded best model epoch {ckpt['epoch']}")
else:
    print("[WARN] No best model found, using current weights.")

print("\n--- LOADING TEST SET ---")
df_test = load_dataset(
    "chuongdo1104/amazon-2023-bronze",
    data_dir="bronze/bronze_test.parquet",
    split="train"
).to_pandas()
u_col = next(c for c in ["user_idx","user_id","reviewer_id"] if c in df_test.columns)
i_col = next(c for c in ["item_idx","item_id","parent_asin"] if c in df_test.columns)
if df_test[u_col].dtype == object:
    print("[INFO] Mapping string IDs to integer IDs...")
    df_user_map = pd.read_parquet(download_hf("gold/gold_user_id_map.parquet"))
    df_item_map = pd.read_parquet(download_hf("gold/gold_item_id_map.parquet"))
    u_map = dict(zip(df_user_map["reviewer_id"], df_user_map["user_idx"]))
    i_map = dict(zip(df_item_map["parent_asin"], df_item_map["item_idx"]))
    df_test["user_idx"] = df_test[u_col].map(u_map)
    df_test["item_idx"] = df_test[i_col].map(i_map)
    df_test = df_test.dropna(subset=["user_idx","item_idx"])

test_users_t = torch.tensor(df_test["user_idx"].values.astype(int), dtype=torch.long)
test_items_t = torch.tensor(df_test["item_idx"].values.astype(int), dtype=torch.long)

# Test positives: giữ tất cả, kể cả cold items.
# Chỉ lọc user/item ngoài range hợp lệ.
base_mask = (
    (test_users_t >= 0) & (test_users_t < num_users) &
    (test_items_t >= 0) & (test_items_t < num_items)
)
TEST_SIZE_RAW  = int(len(test_users_t))
test_users_filt = test_users_t[base_mask].to(device)
test_items_filt = test_items_t[base_mask].to(device)
cold_test_cnt   = int((item_train_freq_cpu[test_items_filt.cpu()] == 0).sum())
print(f"[FILTER] Test: {TEST_SIZE_RAW:,} -> {len(test_users_filt):,} (gồm {cold_test_cnt:,} cold positives)")

print("\n--- TEST MASKING ---")

# train_user_dict: dùng khi đánh giá validation
# test_mask_dict: dùng khi đánh giá test, gồm train + validation items
# Lưu ý: tuyệt đối không đưa test item vào mask_dict, vì test item là đáp án cần tìm.

train_user_dict = {
    int(u): items.detach().cpu().long()
    for u, items in train_mask_dict.items()
}

df_val_mask = pd.DataFrame({
    "u": val_edges_t[0].detach().cpu().numpy(),
    "i": val_edges_t[1].detach().cpu().numpy(),
})

val_user_dict = df_val_mask.groupby("u")["i"].apply(
    lambda x: torch.tensor(x.values, dtype=torch.long, device="cpu")
).to_dict()
val_user_dict = {
    int(u): items.detach().cpu().long()
    for u, items in val_user_dict.items()
}
del df_val_mask

test_mask_dict = {
    u: items.clone()
    for u, items in train_user_dict.items()
}

for u, val_items in val_user_dict.items():
    if u in test_mask_dict:
        test_mask_dict[u] = torch.unique(torch.cat([test_mask_dict[u], val_items]))
    else:
        test_mask_dict[u] = val_items.clone()

print(f"[OK] Train mask dict      : {len(train_user_dict):,} users.")
print(f"[OK] Validation mask dict : {len(val_user_dict):,} users.")
print(f"[OK] Test mask dict       : {len(test_mask_dict):,} users. Test mask = train + validation.")

item_groups_eval = item_pop_group_t.to(device)

val_cold_count  = int((item_train_freq_cpu[val_edges_t[1].cpu()] == 0).sum().item())
test_cold_count = int((item_train_freq_cpu[test_items_filt.cpu()] == 0).sum().item())
print(f"[INFO] Cold positives in validation : {val_cold_count:,}")
print(f"[INFO] Cold positives in test       : {test_cold_count:,}")
print(f"[INFO] Warm items in catalog        : {num_warm_items:,}")
print(f"[INFO] Cold items in catalog        : {num_cold_items:,} (kept in candidate set)")

gc.collect(); torch.cuda.empty_cache()


--- LOAD BEST MODEL ---
[OK] Loaded best model epoch 50

--- LOADING TEST SET ---


bronze/bronze_test.parquet/part-00000-0c(…):   0%|          | 0.00/32.1M [00:00<?, ?B/s]

bronze/bronze_test.parquet/part-00001-0c(…):   0%|          | 0.00/31.9M [00:00<?, ?B/s]

bronze/bronze_test.parquet/part-00002-0c(…):   0%|          | 0.00/64.0M [00:00<?, ?B/s]

bronze/bronze_test.parquet/part-00003-0c(…):   0%|          | 0.00/63.9M [00:00<?, ?B/s]

...

[INFO] Mapping string IDs to integer IDs...
[INFO] Downloading gold/gold_user_id_map.parquet...


gold/gold_user_id_map.parquet:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

[INFO] Downloading gold/gold_item_id_map.parquet...


gold/gold_item_id_map.parquet:   0%|          | 0.00/97.0M [00:00<?, ?B/s]

[FILTER] Test: 1,847,662 -> 1,847,662 (gồm 89,909 cold positives)

--- TEST MASKING ---
[OK] Train mask dict      : 1,847,662 users.
[OK] Validation mask dict : 1,847,662 users.
[OK] Test mask dict       : 1,847,662 users. Test mask = train + validation.
[INFO] Cold positives in validation : 72,685
[INFO] Cold positives in test       : 89,909
[INFO] Warm items in catalog        : 1,042,121
[INFO] Cold items in catalog        : 567,891 (kept in candidate set)


In [17]:
print("\n--- BUILDING FINAL REPRESENTATIONS ---")
model.eval()
gc.collect(); torch.cuda.empty_cache()

with torch.no_grad():
    with torch.amp.autocast("cuda", enabled=(device=="cuda")):
        z_G_all = model.forward_gcn(sparse_adj)
    user_G_all, item_G_all = z_G_all.split([num_users, num_items])
    item_final_all = item_G_all
    user_final_all = user_G_all.cpu()
    del z_G_all, user_G_all, item_G_all

gc.collect(); torch.cuda.empty_cache()
print("[OK] Representations ready.")

# Flat arrays — interaction-level: mỗi (user, item) là một mẫu độc lập
test_u_idx = test_users_filt.cpu().numpy().astype("int64")
test_i_idx = test_items_filt.cpu().numpy().astype("int64")
print(f"[TEST] {len(test_u_idx):,} interactions | Items: {num_items:,} (warm={num_warm_items:,} cold={num_cold_items:,})")

POP_NAMES_TEST = {2: "Item_TAIL", 3: "Item_COLD"}


def run_final_evaluation(Ks=(20, 40), user_batch=128):
    """
    Full-ranking INTERACTION-LEVEL test evaluation.

    Protocol:
    - Mỗi (user, positive_item) là một mẫu đánh giá độc lập.
    - Candidate set: toàn bộ catalog (kể cả cold items).
    - Mask: train + validation items bị loại khỏi ranking.
    - Recall@K / NDCG@K: trung bình qua toàn bộ interactions.
    - TAIL/COLD: lọc interactions có positive item thuộc nhóm đó.
    """
    Ks   = sorted(list(Ks))
    maxK = max(Ks)
    n_test = len(test_u_idx)

    item_groups_dev = item_pop_group_t.to(device)
    group_names     = ["OVERALL", "Item_TAIL", "Item_COLD"]
    accum = {
        g: {K: {"hit": 0.0, "ndcg": 0.0, "n": 0} for K in Ks}
        for g in group_names
    }
    recommended_mask = {
        K: torch.zeros(num_items, dtype=torch.bool, device=device) for K in Ks
    }
    tail_mask_eval = (item_groups_dev == 2)
    mask_value     = torch.finfo(torch.float32).min

    print(f"\n--- FULL-RANKING TEST INTERACTION-LEVEL (Ks={Ks}) ---")
    print(f"Test: {n_test:,} interactions | Items: {num_items:,} (warm={num_warm_items:,} cold={num_cold_items:,})")

    for start in tqdm(range(0, n_test, user_batch), desc="Test Eval", ncols=100):
        end    = min(start + user_batch, n_test)
        bu_np  = test_u_idx[start:end]
        bi_np  = test_i_idx[start:end]
        bu_cpu = torch.tensor(bu_np, dtype=torch.long)
        bi     = torch.tensor(bi_np, dtype=torch.long, device=device)

        u_embs = user_final_all[bu_cpu].to(device)

        with torch.no_grad():
            with torch.amp.autocast("cuda", enabled=(device=="cuda")):
                scores = torch.mm(u_embs, item_final_all.t())

        # Mask train + validation items (không mask test positives)
        for j, uid in enumerate(bu_np):
            seen = test_mask_dict.get(int(uid))
            if seen is not None:
                scores[j, seen.to(device)] = mask_value

        # Rank của positive item (1-indexed)
        pos_sc = scores[torch.arange(len(bu_cpu), device=device), bi]
        ranks  = (scores > pos_sc.unsqueeze(1)).sum(dim=1) + 1

        # Coverage mask
        _, topk_idx = torch.topk(scores, maxK, dim=1)
        for K in Ks:
            recommended_mask[K][topk_idx[:, :K].reshape(-1)] = True

        # Tích lũy metrics
        bi_g = item_groups_dev[bi]
        for K in Ks:
            hm = (ranks <= K).float()
            nm = torch.where(
                ranks <= K,
                1.0 / torch.log2(ranks.float() + 1.0),
                torch.zeros_like(ranks.float())
            )
            accum["OVERALL"][K]["hit"]  += hm.sum().item()
            accum["OVERALL"][K]["ndcg"] += nm.sum().item()
            accum["OVERALL"][K]["n"]    += len(bu_np)

            for code, gn in POP_NAMES_TEST.items():
                m = (bi_g == code)
                if m.any():
                    accum[gn][K]["hit"]  += hm[m].sum().item()
                    accum[gn][K]["ndcg"] += nm[m].sum().item()
                    accum[gn][K]["n"]    += m.sum().item()

        del u_embs, scores

    # Aggregate
    results = {}
    for g in group_names:
        results[g] = {}
        for K in Ks:
            n = accum[g][K]["n"]
            results[g][f"Recall@{K}"] = accum[g][K]["hit"]  / max(n, 1)
            results[g][f"NDCG@{K}"]   = accum[g][K]["ndcg"] / max(n, 1)
            results[g][f"n@{K}"]      = n

    # Coverage
    n_tail = int(tail_mask_eval.sum().item())
    cov, tcov, apop = {}, {}, {}
    for K in Ks:
        rm  = recommended_mask[K]
        nr  = int(rm.sum().item())
        cov[K]  = nr / num_items if num_items > 0 else 0
        tcov[K] = (rm & tail_mask_eval).sum().item() / n_tail if n_tail > 0 else 0
        apop[K] = item_train_freq_t[rm].float().mean().item() if nr > 0 else 0

    results["CoverageByK"]      = cov
    results["TailCoverageByK"]  = tcov
    results["AvgPopularityByK"] = apop
    results["EvalLevel"]        = "interaction-level"

    # Print
    print("\n" + "="*90)
    print("FINAL TEST — FULL-RANKING INTERACTION-LEVEL")
    print("="*90)
    for g in ["OVERALL", "Item_TAIL", "Item_COLD"]:
        if g not in results: continue
        n = results[g].get(f"n@{Ks[0]}", 0)
        parts = [f"{g:<16} | n={n:>10,}"]
        for K in Ks:
            r  = results[g].get(f"Recall@{K}", 0)
            nd = results[g].get(f"NDCG@{K}",   0)
            parts.append(f"R@{K}={r:.4f} N@{K}={nd:.4f}")
        print(" | ".join(parts))
    for K in Ks:
        print(f"Coverage@{K}={cov[K]:.4f} | TailCov@{K}={tcov[K]:.4f} | AvgPop@{K}={apop[K]:.1f}")
    tail_r20 = results.get("Item_TAIL", {}).get("Recall@20", 0)
    tail_n20 = results.get("Item_TAIL", {}).get("NDCG@20",   0)
    cold_r20 = results.get("Item_COLD", {}).get("Recall@20", 0)
    cold_n20 = results.get("Item_COLD", {}).get("NDCG@20",   0)
    print(f"\n  [TAIL] R@20={tail_r20:.4f} N@20={tail_n20:.4f} TailCov@20={tcov.get(20,0):.4f}")
    print(f"  [COLD] R@20={cold_r20:.4f} N@20={cold_n20:.4f} (expected ~0 for Pure CF baseline)")
    print("="*90)
    return results


results_full = run_final_evaluation(Ks=(20, 40))


--- BUILDING FINAL REPRESENTATIONS ---
[OK] Representations ready.
[TEST] 1,847,662 interactions | Items: 1,610,012 (warm=1,042,121 cold=567,891)

--- FULL-RANKING TEST INTERACTION-LEVEL (Ks=[20, 40]) ---
Test: 1,847,662 interactions | Items: 1,610,012 (warm=1,042,121 cold=567,891)


Test Eval:   0%|                                                          | 0/14435 [00:00<?, ?it/s]


FINAL TEST — FULL-RANKING INTERACTION-LEVEL
OVERALL          | n= 1,847,662 | R@20=0.0183 N@20=0.0074 | R@40=0.0280 N@40=0.0094
Item_TAIL        | n=   173,508 | R@20=0.0000 N@20=0.0000 | R@40=0.0000 N@40=0.0000
Item_COLD        | n=    89,909 | R@20=0.0000 N@20=0.0000 | R@40=0.0000 N@40=0.0000
Coverage@20=0.0010 | TailCov@20=0.0011 | AvgPop@20=684.8
Coverage@40=0.0018 | TailCov@40=0.0021 | AvgPop@40=495.4

  [TAIL] R@20=0.0000 N@20=0.0000 TailCov@20=0.0011
  [COLD] R@20=0.0000 N@20=0.0000 (expected ~0 for Pure CF baseline)


In [18]:
import json, time
import numpy as np

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

print("="*70)
print(f"{'FINAL REPORT — ' + VARIANT_NAME:^70}")
print("="*70)

final_metrics = {
    "variant":                VARIANT_NAME,
    "timestamp":              time.strftime("%Y-%m-%d %H:%M:%S"),
    "protocol":               "full_ranking_interaction_level",
    "eval_level":             "interaction-level",
    "ks":                     [20, 40],
    "num_items_total":        int(num_items),
    "num_warm_items":         int(num_warm_items),
    "num_cold_items":         int(num_cold_items),
    "val_size_interactions":  int(VAL_SIZE),
    "test_size_interactions": int(len(test_users_filt)),
}

if "results_full" in globals() and isinstance(results_full, dict):
    for group_name in ["OVERALL", "Item_TAIL", "Item_COLD"]:
        if group_name in results_full:
            for K in [20, 40]:
                final_metrics[f"{group_name}_Recall@{K}"] = results_full[group_name].get(f"Recall@{K}", 0)
                final_metrics[f"{group_name}_NDCG@{K}"]   = results_full[group_name].get(f"NDCG@{K}", 0)
                final_metrics[f"{group_name}_n@{K}"]      = results_full[group_name].get(f"n@{K}", 0)

    for K in [20, 40]:
        final_metrics[f"Coverage@{K}"]      = results_full.get("CoverageByK", {}).get(K, 0)
        final_metrics[f"TailCoverage@{K}"]  = results_full.get("TailCoverageByK", {}).get(K, 0)
        final_metrics[f"AvgPopularity@{K}"] = results_full.get("AvgPopularityByK", {}).get(K, 0)

with open(os.path.join(CFG["DATA_DIR"], f"report_{VARIANT_NAME}.json"), "w") as f:
    json.dump(final_metrics, f, indent=2, cls=NpEncoder)
print(f"[OK] Report saved to {CFG['DATA_DIR']}/report_{VARIANT_NAME}.json")

                   FINAL REPORT — lightgcn_baseline                   
[OK] Report saved to /content/drive/MyDrive/tarecmind/data_lightgcn_baseline/report_lightgcn_baseline.json


In [19]:
import time
print("Pipeline complete. Disconnecting in 10s...")
time.sleep(10)
try:
    from google.colab import runtime
    runtime.unassign()
except:
    print("[INFO] Not in Colab.")

Pipeline complete. Disconnecting in 10s...
